# Privacy Auditing: Measuring Empirical Privacy Leakage

DP proofs guarantee privacy in theory. Privacy auditing tests whether the guarantee holds empirically and measures how tight the formal bound is in practice.

## LiRA: Likelihood Ratio Attack

LiRA (Carlini et al. 2022) is the state-of-the-art membership inference attack, used to empirically estimate the true ε of a trained model:

1. Train many shadow models — some with a target record (member), some without (non-member)
2. For each shadow model, compute the model's confidence on the target record
3. Fit two Gaussians to the confidence distributions (member vs non-member)
4. The likelihood ratio between these distributions gives the best possible distinguishing advantage
5. Convert the advantage to an empirical ε

If the empirical ε is much lower than the formal ε, the formal bound is loose (which is good — the model is more private than guaranteed). If empirical ε exceeds formal ε, the formal analysis may have an error.

In [ ]:
import math, random
from dataclasses import dataclass

@dataclass
class LiRAResult:
    target_record: int
    empirical_epsilon: float
    formal_epsilon: float
    attack_tpr_at_low_fpr: float
    n_shadow_models: int

def simulate_lira(
    target_record_id: int,
    n_shadow_models: int = 256,
    formal_epsilon: float = 3.0,
    seed: int = 42
) -> LiRAResult:
    rng = random.Random(seed)
    # Member shadow models: target in training set
    member_confs = [min(1.0, max(0.01, rng.gauss(0.82, 0.08))) for _ in range(n_shadow_models//2)]
    # Non-member shadow models
    nonmember_confs = [min(1.0, max(0.01, rng.gauss(0.68, 0.10))) for _ in range(n_shadow_models//2)]
    # Fit Gaussians
    def mean_std(data):
        mu = sum(data)/len(data)
        std = math.sqrt(sum((x-mu)**2 for x in data)/len(data))
        return mu, max(0.001, std)
    mu_m, std_m = mean_std(member_confs)
    mu_nm, std_nm = mean_std(nonmember_confs)
    # LR test on threshold sweep
    def log_likelihood(x, mu, std):
        return -0.5 * ((x - mu) / std)**2 - math.log(std)
    all_confs = sorted(member_confs + nonmember_confs)
    best_tpr_at_1pct_fpr = 0
    for threshold in all_confs:
        fpr = sum(1 for c in nonmember_confs if c > threshold) / len(nonmember_confs)
        tpr = sum(1 for c in member_confs if c > threshold) / len(member_confs)
        if fpr <= 0.01:
            best_tpr_at_1pct_fpr = max(best_tpr_at_1pct_fpr, tpr)
    # Empirical epsilon from TPR/FPR at low FPR
    if best_tpr_at_1pct_fpr > 0.01:
        empirical_eps = math.log(best_tpr_at_1pct_fpr / 0.01)
    else:
        empirical_eps = 0.0
    return LiRAResult(target_record=target_record_id, empirical_epsilon=empirical_eps,
                       formal_epsilon=formal_epsilon, attack_tpr_at_low_fpr=best_tpr_at_1pct_fpr,
                       n_shadow_models=n_shadow_models)

# Test on records from different parts of the distribution
for record_id in [1, 42, 100]:
    result = simulate_lira(record_id, n_shadow_models=200, formal_epsilon=3.0)
    print(f'Record {record_id}:')
    print(f'  Formal ε: {result.formal_epsilon:.2f}')
    print(f'  Empirical ε: {result.empirical_epsilon:.4f}')
    print(f'  TPR @ 1% FPR: {result.attack_tpr_at_low_fpr:.4f}')
    print(f'  Audit result: {"PASS - formal bound holds" if result.empirical_epsilon <= result.formal_epsilon else "FAIL - leakage exceeds formal bound"}')

## Continuous Privacy Monitoring

Privacy auditing should be part of the ML training pipeline, not a one-time check:
1. After each training run, compute empirical ε via LiRA on a held-out audit set
2. Log empirical ε alongside model accuracy — track both over time
3. Alert if empirical ε approaches or exceeds the formal bound
4. For high-stakes deployments, require audit before production release

This turns privacy from a theoretical guarantee into a measured, monitored property.